## LEAK SYNTHETIC DATA GENRATION FOR FAULT TRAINING

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import os

# Reproducibility
np.random.seed(42)

In [2]:
# Pipe Geometry
pipe_length = 50.0  # meters
pipe_diameter = 0.2 # meters
pipe_radius = pipe_diameter / 2
pipe_area = np.pi * pipe_radius**2  # m^2

# Nodes per timestep
nodes_per_timestep = 146
timesteps = 700

# Normal operations baseline values
baseline = {
    'pressure':           19657.0,
    'pressure-coefficient': 32600.0,
    'density':            2700.0,
    'velocity-magnitude': 2.479,
    'x-velocity':         2.478,
    'y-velocity':         0.0017,
    'z-velocity':        -0.0013,
    'turb-kinetic-energy': 0.0363,
    'turb-diss-rate':     24.4,
    'wall-shear':         10.4,
    'tailings-vof':       0.400,
}

# Normal operation std devs for noise injection
noise = {
    'pressure':           500.0,
    'pressure-coefficient': 800.0,
    'density':            0.0,     # constant
    'velocity-magnitude': 0.05,
    'x-velocity':         0.05,
    'y-velocity':         0.005,
    'z-velocity':         0.005,
    'turb-kinetic-energy': 0.002,
    'turb-diss-rate':     1.5,
    'wall-shear':         0.5,
    'tailings-vof':       0.005,
}

# Leak locations and Severity
leak_locations = [10.0, 20.0, 25.0, 35.0, 40.0]
leak_severity = {
        'incipient': (6,  0.25),
        'moderate':  (10, 0.55),
        'critical':  (14, 0.90),
}

# Spatial decay lengths
upstream_decay = 15 * pipe_diameter
downstream_decay = 40 * pipe_diameter

In [3]:
# Output directory
base_dir = Path(os.path.dirname(os.getcwd())) / "data" / "synthetic"
Output_dir = base_dir / "leakage_variations"
Output_dir.mkdir(parents=True, exist_ok=True)


print("Generating synthetic dataset for leak and blockage scenarios...")
print("Total samples per scenario: ", nodes_per_timestep * timesteps)
print("Total scenarios: ", len(leak_locations) * len(leak_severity))  # +1 for normal
print("Total leak locations: ", len(leak_locations))
print("Total severity levels: ", len(leak_severity))
print(Output_dir)

Generating synthetic dataset for leak and blockage scenarios...
Total samples per scenario:  102200
Total scenarios:  15
Total leak locations:  5
Total severity levels:  3
/home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/leakage_variations


In [4]:
# Generate node coordinates
def generate_node_coordinates(n_nodes=nodes_per_timestep):
    """Generates node coordinates along the pipe length and circumference."""
    np.random.seed(42)  # For reproducibility

    node_numbers = np.arange(1, n_nodes + 1)

    # X coordinates - distributed along pipe length
    x_coords = np.linspace(0, pipe_length, n_nodes)

    # y and z - radial positions within pipe cross-section, most walls in interior, some near wall
    angles = np.linspace(0, 2 * np.pi, n_nodes)
    radii = np.random.uniform(0, pipe_radius, n_nodes)

    y_coords = radii * np.cos(angles)
    z_coords = radii * np.sin(angles)

    return node_numbers, x_coords, y_coords, z_coords

node_numbers, x_coords, y_coords, z_coords = generate_node_coordinates()
print(f"x range: {x_coords.min():.2f} to {x_coords.max():.2f} m")
print(f"y range: {y_coords.min():.4f} to {y_coords.max():.4f} m")
print(f"z range: {z_coords.min():.4f} to {z_coords.max():.4f} m")

x range: 0.00 to 50.00 m
y range: -0.0976 to 0.0950 m
z range: -0.0920 to 0.0961 m


In [5]:
# Pressure Profile Generator
def generate_normal_pressure_profile(x_coords):
    """
Generates a normal pressure profile along the pipe length with noise.
    """
    p_inlet = 40000.0
    p_outlet = 0.0

    #Linear pressure gradient along pipe
    pressure = p_inlet - (p_inlet - p_outlet) * (x_coords / pipe_length)

    return pressure

def apply_leak_pressure_disturbance(pressure, x_coords, leak_x, effect_factor):
    """
    Physically correct model for pressurized pipeline leak.

    A leak diverts flow laterally. Downstream of the leak,
    reduced flow means reduced friction → pressure deficit
    grows linearly with downstream distance.

    Upstream: slight back-pressure effect (minor).
    Downstream: linear pressure deficit — visible at ALL downstream sensors.
    """
    pressure = pressure.copy()

    # Fraction of flow lost at leak point
    # incipient=0.25 loses ~5%, moderate=0.55 loses ~11%, critical=0.90 loses ~18%
    flow_loss_fraction = 0.20 * effect_factor

    # Normal pressure gradient (Pa per meter) for downstream section
    # Normal: 40,000 Pa over 50m = 800 Pa/m
    normal_gradient = 40000.0 / pipe_length  # 800 Pa/m

    # Pressure drop reduction factor downstream
    # Less flow → friction proportional to v² → gradient scales with (1-k)²
    k = flow_loss_fraction
    gradient_reduction = normal_gradient * k * (2 - k)

    for i, x in enumerate(x_coords):
        distance = x - leak_x

        if distance < 0:
            # UPSTREAM — minor back pressure effect
            # Flow tries to compensate, small upstream pressure rise
            upstream_dist = abs(distance)
            upstream_effect = 0.03 * effect_factor * 8000.0 * \
                              np.exp(-upstream_dist / (10 * pipe_diameter))
            pressure[i] += upstream_effect

        else:
            # DOWNSTREAM — LINEAR pressure deficit
            # Grows with distance from leak → far sensors see MORE signal
            pressure_deficit = gradient_reduction * distance
            pressure[i] -= pressure_deficit

    return pressure

In [6]:
# Velocity disturbance generator
def apply_leak_velocity_disturbance(velocity_mag, x_velocity,
                                     y_velocity, z_velocity,
                                     x_coords, leak_x, effect_factor):
    """
    Leak causes:
    - Local velocity INCREASE at leak (fluid rushing toward opening)
    - Velocity REDUCTION downstream (less flow remains in pipe)
    """
    velocity_mag = velocity_mag.copy()
    x_velocity   = x_velocity.copy()
    y_velocity   = y_velocity.copy()
    z_velocity   = z_velocity.copy()

    max_vel_increase    = 0.8 * effect_factor   # m/s at leak point
    max_vel_decrease    = 0.6 * effect_factor   # m/s downstream

    for i, x in enumerate(x_coords):
        distance = x - leak_x

        if abs(distance) < 0.2:
            # AT LEAK — velocity increases locally
            local_effect = max_vel_increase * np.exp(
                -abs(distance) / (pipe_diameter))
            velocity_mag[i] += local_effect
            x_velocity[i]   += local_effect * 0.7
            # Radial components increase (fluid exits radially)
            y_velocity[i]   += local_effect * 0.15
            z_velocity[i]   -= local_effect * 0.15

        elif distance > 0:
            # DOWNSTREAM — velocity decreases
            downstream_effect = max_vel_decrease * np.exp(
                -distance / downstream_decay)
            velocity_mag[i] = max(0, velocity_mag[i] - downstream_effect)
            x_velocity[i]   = max(0, x_velocity[i]   - downstream_effect)

    return velocity_mag, x_velocity, y_velocity, z_velocity

In [7]:
# Turbulence and tailings disturbance generator
def apply_leak_turbulence_disturbance(tke, tdr, x_coords,
                                       leak_x, effect_factor):
    """
    Leak causes turbulence spike at and downstream of leak point.
    """
    tke = tke.copy()
    tdr = tdr.copy()

    max_tke_increase = 0.015 * effect_factor
    max_tdr_increase = 30.0  * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - leak_x
        if distance >= -0.2:
            decay_dist = max(0, distance + 0.2)
            effect = np.exp(-decay_dist / downstream_decay)
            tke[i] += max_tke_increase * effect
            tdr[i] += max_tdr_increase * effect

    return tke, tdr


def apply_leak_tailings_disturbance(tailings_vof, x_coords,
                                     leak_x, effect_factor):

    tailings_vof = tailings_vof.copy()

    max_drop      = 0.08 * effect_factor   # at leak point
    max_increase  = 0.05 * effect_factor   # downstream

    for i, x in enumerate(x_coords):
        distance = x - leak_x

        if abs(distance) < 0.2:
            # AT LEAK — solids fraction drops
            local_effect = max_drop * np.exp(
                -abs(distance) / pipe_diameter)
            tailings_vof[i] = max(0.163, tailings_vof[i] - local_effect)

        elif distance > 0:
            # DOWNSTREAM — solids concentrate
            downstream_effect = max_increase * np.exp(
                -distance / downstream_decay)
            tailings_vof[i] = min(0.508,
                                  tailings_vof[i] + downstream_effect)

    return tailings_vof

In [8]:
# Wall shear disturbance generator
def apply_leak_wall_shear_disturbance(wall_shear, x_coords,
                                       leak_x, effect_factor):
    wall_shear = wall_shear.copy()

    max_increase = 15.0 * effect_factor
    max_decrease =  5.0 * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - leak_x

        if abs(distance) < 0.3:
            local_effect = max_increase * np.exp(
                -abs(distance) / pipe_diameter)
            wall_shear[i] += local_effect

        elif distance > 0:
            downstream_effect = max_decrease * np.exp(
                -distance / downstream_decay)
            wall_shear[i] = max(0, wall_shear[i] - downstream_effect)

    return wall_shear

In [9]:
# Single timestep generator
def generate_leak_timestep(timestep, leak_x, effect_factor,
                            node_numbers, x_coords,
                            y_coords, z_coords):
    """
    Generate one timestep of leak scenario data.
    Applies all physics disturbances on top of normal baseline.
    """
    n = len(x_coords)

    # ── Start with normal baseline ─────────────────────────
    pressure    = generate_normal_pressure_profile(x_coords)
    pressure   += np.random.normal(0, noise['pressure'], n)

    density     = np.full(n, baseline['density'])

    vel_mag     = np.full(n, baseline['velocity-magnitude']) + \
                  np.random.normal(0, noise['velocity-magnitude'], n)
    x_vel       = np.full(n, baseline['x-velocity']) + \
                  np.random.normal(0, noise['x-velocity'], n)
    y_vel       = np.full(n,baseline['y-velocity']) + \
                  np.random.normal(0, noise['y-velocity'], n)
    z_vel       = np.full(n, baseline['z-velocity']) + \
                  np.random.normal(0, noise['z-velocity'], n)

    tke         = np.full(n, baseline['turb-kinetic-energy']) + \
                  np.random.normal(0, noise['turb-kinetic-energy'], n)
    tdr         = np.full(n, baseline['turb-diss-rate']) + \
                  np.random.normal(0, noise['turb-diss-rate'], n)
    wall_shear  = np.full(n, baseline['wall-shear']) + \
                  np.random.normal(0, noise['wall-shear'], n)
    tailings    = np.full(n, baseline['tailings-vof']) + \
                  np.random.normal(0, noise['tailings-vof'], n)

    # ── Apply leak disturbances ────────────────────────────
    pressure    = apply_leak_pressure_disturbance(
                    pressure, x_coords, leak_x, effect_factor)

    vel_mag, x_vel, y_vel, z_vel = apply_leak_velocity_disturbance(
                    vel_mag, x_vel, y_vel, z_vel,
                    x_coords, leak_x, effect_factor)

    tke, tdr    = apply_leak_turbulence_disturbance(
                    tke, tdr, x_coords, leak_x, effect_factor)

    tailings    = apply_leak_tailings_disturbance(
                    tailings, x_coords, leak_x, effect_factor)

    wall_shear  = apply_leak_wall_shear_disturbance(
                    wall_shear, x_coords, leak_x, effect_factor)

    # ── Clip to physical bounds ────────────────────────────
    pressure    = np.clip(pressure,   0,     45000)
    vel_mag     = np.clip(vel_mag,    0,     3.5)
    x_vel       = np.clip(x_vel,      0,     3.5)
    tke         = np.clip(tke,        0.023, 0.05)
    tdr         = np.clip(tdr,        0.07,  90.0)
    wall_shear  = np.clip(wall_shear, 0,     50.0)
    tailings    = np.clip(tailings,   0.163, 0.508)

    # ── Pressure coefficient ───────────────────────────────
    p_coeff     = pressure * (66166 / 40473)  # scale proportionally

    # ── Build DataFrame ────────────────────────────────────
    df = pd.DataFrame({
        'nodenumber':          node_numbers,
        'x-coordinate':        x_coords,
        'y-coordinate':        y_coords,
        'z-coordinate':        z_coords,
        'pressure':            pressure,
        'pressure-coefficient': p_coeff,
        'density':             density,
        'velocity-magnitude':  vel_mag,
        'x-velocity':          x_vel,
        'y-velocity':          y_vel,
        'z-velocity':          z_vel,
        'turb-kinetic-energy': tke,
        'turb-diss-rate':      tdr,
        'wall-shear':          wall_shear,
        'tailings-vof':        tailings,
        'timestep':            timestep,
        'label':               1,          # LEAK = 1
    })

    return df

In [10]:
# Full leak case generator
def generate_leak_case(leak_location_x, severity_name,
                        effect_factor, save=True):
    """
    Generate full dataset for one leak case.
    700 timesteps × 146 nodes = 102,200 rows.
    """
    node_numbers, x_coords, y_coords, z_coords = \
        generate_node_coordinates()

    all_timesteps = []

    for t in range(1, timesteps + 1):
        df_t = generate_leak_timestep(
            timestep     = t,
            leak_x       = leak_location_x,
            effect_factor = effect_factor,
            node_numbers  = node_numbers,
            x_coords      = x_coords,
            y_coords      = y_coords,
            z_coords      = z_coords,
        )
        all_timesteps.append(df_t)

        if t % 100 == 0:
            print(f"  Timestep {t}/{timesteps} done")

    df_case = pd.concat(all_timesteps, ignore_index=True)

    if save:
        loc_label = int(leak_location_x)
        filename  = f"leak_{severity_name}_loc{loc_label}m.csv"
        filepath  = Output_dir / filename
        df_case.to_csv(filepath, index=False)
        print(f"Saved: {filepath}  shape={df_case.shape}")

    return df_case

In [11]:
# Running though all leak cases
print("=" * 60)
print("GENERATING ALL LEAK SCENARIOS")
print("=" * 60)
print(f"Locations  : {leak_locations}")
print(f"Severities : {list(leak_severity.keys())}")
print(f"Total cases: {len(leak_locations) * len(leak_severity)}")
print(f"Rows/case  : {nodes_per_timestep * timesteps:,}")
print("=" * 60)

leak_dataframes = {}

for loc_x in leak_locations:
    for severity_name, (diameter_mm, effect_factor) in \
            leak_severity.items():

        case_key = f"leak_{severity_name}_{int(loc_x)}m"

        print(f"\nGenerating: {case_key}")
        print(f"  Location  : x = {loc_x} m")
        print(f"  Severity  : {severity_name}")
        print(f"  Diameter  : {diameter_mm} mm")
        print(f"  Effect    : {effect_factor}")

        df_case = generate_leak_case(
            leak_location_x = loc_x,
            severity_name   = severity_name,
            effect_factor   = effect_factor,
            save            = True,
        )

        leak_dataframes[case_key] = df_case

print("\n" + "=" * 60)
print("ALL LEAK CASES GENERATED")
print(f"Total cases saved: {len(leak_dataframes)}")
print("=" * 60)

GENERATING ALL LEAK SCENARIOS
Locations  : [10.0, 20.0, 25.0, 35.0, 40.0]
Severities : ['incipient', 'moderate', 'critical']
Total cases: 15
Rows/case  : 102,200

Generating: leak_incipient_10m
  Location  : x = 10.0 m
  Severity  : incipient
  Diameter  : 6 mm
  Effect    : 0.25
  Timestep 100/700 done
  Timestep 200/700 done
  Timestep 300/700 done
  Timestep 400/700 done
  Timestep 500/700 done
  Timestep 600/700 done
  Timestep 700/700 done
Saved: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/leakage_variations/leak_incipient_loc10m.csv  shape=(102200, 17)

Generating: leak_moderate_10m
  Location  : x = 10.0 m
  Severity  : moderate
  Diameter  : 10 mm
  Effect    : 0.55
  Timestep 100/700 done
  Timestep 200/700 done
  Timestep 300/700 done
  Timestep 400/700 done
  Timestep 500/700 done
  Timestep 600/700 done
  Timestep 700/700 done
Saved: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/leakage_variations/l

In [12]:
# Leak validation
print("VALIDATION — Checking leak signatures are physically correct")
print("=" * 60)

# Pick one case to validate
sample_case = leak_dataframes['leak_critical_25m']

# Get timestep 350 (middle)
ts_sample = sample_case[sample_case['timestep'] == 350].copy()
ts_sample = ts_sample.sort_values('x-coordinate')

# Find nodes near leak (25m) vs far from leak
near_leak      = ts_sample[abs(ts_sample['x-coordinate'] - 25) < 1.0]
far_upstream   = ts_sample[ts_sample['x-coordinate'] < 10]
far_downstream = ts_sample[ts_sample['x-coordinate'] > 40]

print("\nPressure comparison:")
print(f"  Far upstream   : {far_upstream['pressure'].mean():.1f} Pa")
print(f"  Near leak      : {near_leak['pressure'].mean():.1f} Pa")
print(f"  Far downstream : {far_downstream['pressure'].mean():.1f} Pa")

print("\nVelocity comparison:")
print(f"  Far upstream   : {far_upstream['velocity-magnitude'].mean():.3f} m/s")
print(f"  Near leak      : {near_leak['velocity-magnitude'].mean():.3f} m/s")
print(f"  Far downstream : {far_downstream['velocity-magnitude'].mean():.3f} m/s")

print("\nTailings VOF comparison:")
print(f"  Far upstream   : {far_upstream['tailings-vof'].mean():.4f}")
print(f"  Near leak      : {near_leak['tailings-vof'].mean():.4f}")
print(f"  Far downstream : {far_downstream['tailings-vof'].mean():.4f}")

print("\nExpected physics:")
print("  Pressure   : upstream > near_leak (DROP at leak) > downstream")
print("  Velocity   : near_leak HIGHEST, downstream lowest")
print("  Tailings   : near_leak LOWEST (liquid exits), downstream highest")

VALIDATION — Checking leak signatures are physically correct

Pressure comparison:
  Far upstream   : 36004.6 Pa
  Near leak      : 20248.7 Pa
  Far downstream : 695.5 Pa

Velocity comparison:
  Far upstream   : 2.474 m/s
  Near leak      : 2.412 m/s
  Far downstream : 2.424 m/s

Tailings VOF comparison:
  Far upstream   : 0.4003
  Near leak      : 0.4017
  Far downstream : 0.4054

Expected physics:
  Pressure   : upstream > near_leak (DROP at leak) > downstream
  Velocity   : near_leak HIGHEST, downstream lowest
  Tailings   : near_leak LOWEST (liquid exits), downstream highest


In [13]:
# Combining all the leak cases into one big DataFrame
print("Combining all leak cases into single leak dataset...")

all_leak_dfs = list(leak_dataframes.values())
df_all_leaks = pd.concat(all_leak_dfs, ignore_index=True)

print(f"Combined leak dataset shape: {df_all_leaks.shape}")
print(f"Label distribution: {df_all_leaks['label'].value_counts().to_dict()}")
print(f"Total rows: {len(df_all_leaks):,}")

# Save combined
combined_path = os.path.join(Output_dir, 'all_leaks_combined.csv')
df_all_leaks.to_csv(combined_path, index=False)
print(f"Saved combined leak dataset: {combined_path}")

Combining all leak cases into single leak dataset...
Combined leak dataset shape: (1533000, 17)
Label distribution: {1: 1533000}
Total rows: 1,533,000
Saved combined leak dataset: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/leakage_variations/all_leaks_combined.csv
